In [ ]:

import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


csv_files = glob.glob('*.csv')


# 1. Dataset Load (Updated to our new fully merged file)

df = pd.read_csv("cgpa.csv")
print(f"Loaded File Successfully!\n")


print ("total rows and columns in our dataset is")
print (f"Total Rows: {df.shape[0]}")
print (f"total columns: {df.shape[1]}")

print("columns in our dataset are")
print(df.columns.tolist())

# 2. dataset all values checking
print(" Dataset Info ")
df.info()

# 9. Check required columns for CGPA Regression & Stress Analysis
required_columns = [
    "study_hours_per_day",
    "attendance_percentage",
    "previous_semester_gpa",
    "assignment_score",
    "exam_score",
    "Stress_Score",
    "cgpa"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    print("Missing required columns:", missing_columns)
else:
    print("\nAll required columns are available.")

# 3. dataset missing values checking 
print("missing values")
print(df.isnull().sum())
print("missing values percentage")
print(df.isnull().sum() / len(df) * 100)

# duplicate values checking
print(f"Duplicate values: {df.duplicated().sum()}")
df = df.drop_duplicates()

print("=== Target (CGPA) Summary ===")
print(df["cgpa"].describe())

# Features jin ka average check karna hai (Academic + Wellness)
features = [
    "study_hours_per_day",
    "attendance_percentage",
    "previous_semester_gpa",
    "assignment_score",
    "exam_score",
]

# Required columns ki copy
check_df = df[["cgpa"] + features].copy()

# Feature columns ko numeric mein convert karo
for col in features:
    check_df[col] = pd.to_numeric(check_df[col], errors="coerce")

print("\n=== Dataset Head ===")
print(df.head(3))

eda

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# 2. Fixed target column for Regression (CGPA Prediction)
target_col = "cgpa"

# 3. Check target column
if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' not found!")

# 4. Remove rows with missing target values
clean_df = df.dropna(subset=[target_col]).copy()

# 5. CGPA Distribution Plot (Histogram & KDE)
plt.figure(figsize=(10, 6))
sns.histplot(data=clean_df,
             x=target_col,
             kde=True,
             color="steelblue")
plt.title("Distribution of Student CGPA")
plt.xlabel("CGPA")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# 6. Target summary stats
print("=== CGPA Target Statistics ===")
print(clean_df[target_col].describe())

# 7. Separate input features (X) and target (y)
# Also drop ID or non-predictive string columns if any
drop_cols = [target_col]
if 'student_id' in clean_df.columns:
    drop_cols.append('student_id')

X = clean_df.drop(columns=drop_cols)
y = clean_df[target_col]

print("\nInput Features:")
print(X.columns.tolist())

print("\nTarget Column:", target_col)

# 8. Select numerical columns
num_cols = clean_df.select_dtypes(
    include=[np.number]
).columns.tolist()

# Remove target if it is numeric
if target_col in num_cols:
    num_cols.remove(target_col)

print("\nNumerical Features:", num_cols)

# 9. Histograms of top numerical features
if num_cols:
    cols_to_plot = num_cols[:6]

    clean_df[cols_to_plot].hist(
        bins=20,
        figsize=(12, 8),
        layout=(2, 3),
        color="skyblue",
        edgecolor="black"
    )

    plt.suptitle(
        "Numerical Feature Distributions",
        fontsize=16,
        fontweight="bold"
    )
    plt.tight_layout()
    plt.show()
else:
    print("No numerical input features available for histograms.")

# 10. Correlation Heatmap
num_df = clean_df[num_cols + [target_col]]

if num_df.shape[1] > 1:
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        num_df.corr(),
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        cbar=True
    )
    plt.title("Correlation Heatmap with CGPA")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numerical features for correlation heatmap.")

model training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Select input features and target for CGPA Regression
target_col = "cgpa"

required_cols = [
    "study_hours_per_day",
    "attendance_percentage",
    "previous_semester_gpa",
    "assignment_score",
    "exam_score",
    "sleep_hours"
]

# Check columns
all_cols = required_cols + [target_col]
missing_cols = [col for col in all_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Columns not found: {missing_cols}")

# 2. Select required columns, including target
clean_df = df[all_cols].copy()

# 3. Remove rows with missing target
clean_df = clean_df.dropna(subset=[target_col]).copy()

# 4. Separate input features and target
X = clean_df[required_cols].copy()
Y = clean_df[target_col].copy()

# 5. Encode categorical input features (if any text/category columns exist)
label_encoders = {}

for col in X.select_dtypes(
    include=["object", "category"]
).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# Target is continuous (CGPA), so no label encoding needed for regression target Y
# (If Y was categorical, we'd encode it, but for CGPA regression it's float numbers)

# 6. Fill missing input values
X = X.fillna(X.median(numeric_only=True))

# 7. Split dataset (Note: stratify is removed because CGPA is continuous, not categorical classes)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.30,
    random_state=42
)

# 8. Scale features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 9. Display results
print("Preprocessing & Scaling Complete for CGPA Regression")
print("Total Features:", X.shape[1])
print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

print("\nInput Features:")
print(X.columns.tolist())

print("\nTarget Column:", target_col)
print("\nTarget (CGPA) Summary Statistics:")
print(Y.describe())

importance feature


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

# 1. Train the Random Forest Regressor model for CGPA prediction
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

model.fit(X_train_scaled, y_train)

# 2. Extract feature importances
importances = model.feature_importances_

# 3. Create a DataFrame of feature importances
feat_importances = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": importances
})

# 4. Sort features by importance
feat_importances = feat_importances.sort_values(
    by="Importance",
    ascending=False
)

# 5. Display feature importances
print("=== FEATURE IMPORTANCES (CGPA PREDICTION) ===")
print(feat_importances.to_string(index=False))

# 6. Plot feature importances
top_features = feat_importances.sort_values(
    by="Importance",
    ascending=True
)

plt.figure(figsize=(10, 5))

plt.barh(
    top_features["Feature"],
    top_features["Importance"],
    color="steelblue",
    edgecolor="black"
)

plt.title("Feature Importances - Student CGPA Prediction System")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import pandas as pd
import matplotlib.pyplot as plt

# 1. Regression Models
models = {
    "Baseline Dummy": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "K-Neighbors Regressor": KNeighborsRegressor(n_neighbors=5),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=42)
}

# 2. Store results and trained models
results = []
train_model = {}

# 3. Train and compare models
for name, model in models.items():

    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5  # Clean and modern way to calculate Root Mean Squared Error
    mae = mean_absolute_error(y_test, y_pred)

    results.append({
        "Model": name,
        "R2 Score": r2,
        "RMSE": rmse,
        "MSE": mse,
        "MAE": mae
    })

    # Save each trained model with its name
    train_model[name] = model

    print(f"\n===== {name} Evaluation =====")
    print(f"R2 Score: {r2:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")


# 4. Display model comparison (Sorted by R2 Score descending, higher is better)
results_ans = pd.DataFrame(results)

results_ans = results_ans.sort_values(
    by="R2 Score", ascending=False
).reset_index(drop=True)

print("\n===== REGRESSION MODEL COMPARISON RESULTS =====")
print(results_ans.round(4).to_string(index=False))


# 5. Select the best model
best_model_name = results_ans.loc[0, "Model"]
best_model = train_model[best_model_name]

print("\nBest Model:", best_model_name)
print("Best R2 Score:", round(
    results_ans.loc[0, "R2 Score"], 4
))
print("Best RMSE:", round(
    results_ans.loc[0, "RMSE"], 4
))


# 6. Actual vs Predicted Scatter Plot for the Best Model
y_pred_best = best_model.predict(X_test_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_best, alpha=0.3, color="steelblue", edgecolors="k")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label="Ideal Fit")
plt.title(f"Zaeem Sandhu - Actual vs Predicted CGPA ({best_model_name})")
plt.xlabel("Actual CGPA")
plt.ylabel("Predicted CGPA")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

# 1. Safety Check: Load dataset if df is not defined
if 'df' not in globals():
    df = pd.read_csv("fully_merged_all_columns_dataset.csv")

# 2. Define exact features present in your CSV
target_col = "cgpa"
required_cols = [
    "study_hours_per_day",
    "attendance_percentage",
    "previous_semester_gpa",
    "assignment_score",
    "exam_score"
]

# 3. Prepare Clean DataFrame
clean_df = df[required_cols + [target_col]].dropna(subset=[target_col]).copy()
X = clean_df[required_cols]
Y = clean_df[target_col]

# Fill missing values if any
X = X.fillna(X.median(numeric_only=True))

# 4. Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.30, random_state=42)

# 5. Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Train Best Model (Random Forest Regressor)
best_model = RandomForestRegressor(n_estimators=100, random_state=42)
best_model.fit(X_train_scaled, y_train)

print("✅ Model Trained Successfully with 5 Features!\n")

# 7. Live Test / Prediction
print("="*50)
print("      🎯 STUDENT CGPA PREDICTION (LIVE TEST)")
print("="*50)

# Sample values
study_hours = 4  
attendance = 88.0     
prev_gpa = 3.22    
assignment = 80   
exam = 70           

user_input_dict = {
    "study_hours_per_day": [study_hours],
    "attendance_percentage": [attendance],
    "previous_semester_gpa": [prev_gpa],
    "assignment_score": [assignment],
    "exam_score": [exam]
}

user_df = pd.DataFrame(user_input_dict)

# Scale and Predict
user_scaled = scaler.transform(user_df)
predicted_cgpa = best_model.predict(user_scaled)[0]

# Display Results
print(f"Study Hours / Day    : {study_hours}")
print(f"Attendance           : {attendance}%")
print(f"Previous Sem GPA     : {prev_gpa}")
print(f"Assignment Score     : {assignment}")
print(f"Exam Score           : {exam}")
print("-"*50)
print(f"✨ Predicted Next Sem CGPA: {predicted_cgpa:.2f} / 4.00")
print("="*50)